<a href="https://colab.research.google.com/github/anastasiaalimova/anastasiaalimova.github.io/blob/main/ILCB_Day1_TP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP1: Word prediction in Large Language Models



We need the PyTorch library because it provides the foundation for building and training neural networks, enabling complex mathematical operations essential for both training and inference processes.

In [ ]:
import torch

### Model

We will use the Transformers library by Hugging Face, a powerful toolkit offering pre-trained models for diverse natural language processing (NLP) tasks

You can read about it here:

https://huggingface.co/docs/transformers/index


In [ ]:
import transformers

Within the Transformers library, we utilize AutoModelForMaskedLM, a class designed to load pre-trained models specifically for masked language modeling (MLM).

In MLM, certain words within a sentence are masked, and the model aims to predict these hidden words based on context.


In [ ]:
from transformers import AutoModelForMaskedLM

AutoTokenizer, a class within the Transformers library, automatically loads the appropriate tokenizer corresponding to your chosen model. It breaks down text into units (words or subwords) that the model can process.

In [ ]:
from transformers import AutoTokenizer

We will use the following model:

In [ ]:
Model_name = "phueb/BabyBERTa-1"


Here we load the model and its tokenizer using the classes we introduced above:


In [ ]:
model = AutoModelForMaskedLM.from_pretrained(Model_name)
tokenizer = AutoTokenizer.from_pretrained(Model_name)

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/610 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/34.2M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/328k [00:00<?, ?B/s]

### Input

Using the sentence "The cat sits on the [mask]", we study if the model can predict the masked word ("mat") by leveraging the surrounding context. This example aims to illustrate the fundamental inference process within this class of transformer models.


The tokenizer uses a specific string to represent the masked token. Let's see what it is in this case:

In [ ]:
print("MASK token string:", tokenizer.mask_token)

MASK token string: <mask>


We tokenize the input sentence containing the mask token

In [ ]:
inputs = tokenizer("The cat sits on the <mask>.", return_tensors="pt")

# This will show you the tokenized version of your sentence
print("Tokenized Input:", inputs)


Tokenized Input: {'input_ids': tensor([[   3, 1312, 1357,  190,  881,  245,  187,    0,   18,    4]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1]])}


Compare the number of words in the original sentence to the number of indices in the tokenized input. Do you notice a difference in count? What does this suggest about the tokenization process?


The tokenizer often breaks down words into smaller parts called subwords. Let's convert the input IDs back into tokens to see what tokens the models ended up using:

In [ ]:
tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])

print(tokens)

['<s>', 'the', 'Ġcat', 'Ġs', 'its', 'Ġon', 'Ġthe', '<mask>', '.', '</s>']



The symbol Ġ indicates that a space preceded the token

Why do you think subword tokenization is used in models like these?
Hint: Consider how this helps the model deal with rare words or unfamiliar terms.


### Output

Let's get the model to process the input and generate the output

In [ ]:
with torch.no_grad(): #This line means we only want the model to generate predictions and not update its internal parameters based on this specific example
  outputs = model(**inputs)
predictions = outputs.logits

# This prints the raw logits (the scores before converting them to probabilities) for each token in the sentence.
print("Model Outputs (Logits):", predictions)

Model Outputs (Logits): tensor([[[ -5.1242,  -5.6739,  -4.9963,  ...,  -4.6596,  -3.4309,  -2.8060],
         [ -7.2455,  -9.2986,  -9.2676,  ...,  -8.6715,  -9.2183,  -6.0975],
         [ -6.9107, -11.1869, -11.4115,  ...,  -9.6460, -10.4297,  -8.3409],
         ...,
         [ -8.0542,  -8.6202,  -8.6650,  ...,  -8.3142,  -7.3994,  -7.3036],
         [ -5.0558,  -5.5836,  -4.8891,  ...,  -4.5570,  -3.3704,  -2.7601],
         [ -5.0750,  -5.5437,  -4.8562,  ...,  -4.5393,  -3.3661,  -2.7434]]])


After processing the input, the model generates an output

tensor with a shape of N x V, where:

N is the sequence length (number of tokens in our input sentence) and
V is the vocabulary size (the total number of words the model knows)

In [ ]:
N = outputs.logits.shape[1] # Sequence Length
V = outputs.logits.shape[2] # Vocabulary Size

print(f"rows: Sequence Length (N): {N}")
print(f"columns: Vocabulary Size (V): {V}")


rows: Sequence Length (N): 10
columns: Vocabulary Size (V): 8192


For each token in the input sentence, the model calculates a score (called 'logit') for every possible word in the vocabulary. These scores indicate how likely each word in the vocabulary is to replace the token at that position.  




### Word prediction

We'll focus on the output logits specifically for the masked
token that we want the model to predict:

In [ ]:
# Find the index of the masked token
mask_token_index = torch.where(inputs["input_ids"] == tokenizer.mask_token_id)[1]

# Find the scores for each words in the vocabulary to replace the masked token
mask_token_logits = outputs.logits[0, mask_token_index, :]

print("Logits for Masked Token:", mask_token_logits)

Logits for Masked Token: tensor([[-8.0542, -8.6202, -8.6650,  ..., -8.3142, -7.3994, -7.3036]])


In [ ]:
# Convert logits to probabilities, showing how likely each token is as a replacement for the [mask]

probabilities = torch.softmax(mask_token_logits, dim=1)

print("Probabilities for Each Token:", probabilities)

Probabilities for Each Token: tensor([[3.5863e-07, 2.0361e-07, 1.9470e-07,  ..., 2.7652e-07, 6.9025e-07,
         7.5964e-07]])


Finally, we select the token with the highest probability:

In [ ]:
# Get the top predicted token ID
predicted_token_id = torch.argmax(probabilities, dim=1)

# Decode the predicted ID to get the predicted word
predicted_word = tokenizer.decode(predicted_token_id)
print("Predicted Word:", predicted_word)

Predicted Word:  farm


In [ ]:
# Check if the prediction is correct
correct_word = "mat"
print(f"Correct Word: {correct_word}, Predicted Word: {predicted_word}")

Correct Word: mat, Predicted Word:  farm


What do you think of this prediction? Did the model succeed? Did the model fail?



10 most probable words:

In [ ]:
# Top-k tokens
k = 10
top_k_probs, top_k_token_ids = torch.topk(probabilities, k=k, dim=0)

# Decode token IDs to strings
top_k_tokens = [tokenizer.decode([tid.item()]) for tid in top_k_token_ids]

print("Top-k tokens:", top_k_tokens)
print("Top-k probs:", top_k_probs.tolist())

NameError: name 'torch' is not defined

Why do you think the model does not have "mat" even in the top 10 predictions? Same question: Did the model succeed? Did the model fail? what would the model do if this was encountered during training?





## Homework to return

1) Rerun the same pipeline using a different, unique sentence of your choosing and print the top 10 predictions for the masked word. Aim for a sentence unlikely to be chosen by others in the class.

2) Rerun the same pipelint using your own sentence with the following model:




In [ ]:
Model_name = "roberta-base"

Which model appears more accurate based on your experimentation with various examples? After exploring the documentation for both "roberta-base" and "phueb/BabyBERTa-1" on Hugging Face, propose an explanation for the observed difference.

3) (Optional for advanced students): Adapt the pipeline to function with auto-regressive models, using the following model



In [ ]:
Model_name = "gpt-2"